In [0]:
CREATE OR REPLACE TABLE ecommerce_lakehouse.gold.dim_product
USING DELTA
AS
SELECT
    p.product_id,
    p.product_category_name,
    COALESCE(
        c.product_category_name_english,
        'unknown'
    ) AS product_category_name_english,
    p.product_name_length,
    p.product_description_length,
    p.product_photos_qty,
    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm
FROM ecommerce_lakehouse.silver.products p
LEFT JOIN ecommerce_lakehouse.silver.category_translation c
    ON p.product_category_name =
       c.product_category_name;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE ecommerce_lakehouse.gold.dim_seller
USING DELTA
AS
SELECT
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state
FROM ecommerce_lakehouse.silver.sellers;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE ecommerce_lakehouse.gold.payment_summary
USING DELTA
AS
SELECT
    order_id,
    SUM(payment_value) AS total_payment_value,
    MAX(payment_installments) AS max_installments,
    FIRST(payment_type) AS primary_payment_type
FROM ecommerce_lakehouse.silver.payments
GROUP BY order_id;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE ecommerce_lakehouse.gold.review_summary
USING DELTA
AS
SELECT
    order_id,
    AVG(review_score) AS average_review_score,
    COUNT(*) AS review_count
FROM ecommerce_lakehouse.silver.reviews
GROUP BY order_id;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE ecommerce_lakehouse.gold.fact_order_items
USING DELTA
AS
SELECT
    oi.order_id,
    oi.order_item_id,

    o.customer_id,

    oi.product_id,
    oi.seller_id,

    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    oi.price,
    oi.freight_value,

    oi.price + oi.freight_value AS total_item_value,

    p.product_category_name,
    p.product_category_name_english,

    c.customer_city,
    c.customer_state,

    s.seller_city,
    s.seller_state,

    ps.total_payment_value,
    ps.primary_payment_type,
    ps.max_installments,

    rs.average_review_score,
    rs.review_count

FROM ecommerce_lakehouse.silver.order_items oi

INNER JOIN ecommerce_lakehouse.silver.orders o
    ON oi.order_id = o.order_id

LEFT JOIN ecommerce_lakehouse.gold.dim_product p
    ON oi.product_id = p.product_id

LEFT JOIN ecommerce_lakehouse.silver.customers c
    ON o.customer_id = c.customer_id

LEFT JOIN ecommerce_lakehouse.gold.dim_seller s
    ON oi.seller_id = s.seller_id

LEFT JOIN ecommerce_lakehouse.gold.payment_summary ps
    ON oi.order_id = ps.order_id

LEFT JOIN ecommerce_lakehouse.gold.review_summary rs
    ON oi.order_id = rs.order_id;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE ecommerce_lakehouse.gold.daily_sales
USING DELTA
AS
SELECT
    DATE(order_purchase_timestamp) AS sales_date,

    COUNT(DISTINCT order_id) AS total_orders,

    COUNT(*) AS total_items,

    SUM(price) AS product_revenue,

    SUM(freight_value) AS freight_revenue,

    SUM(total_item_value) AS total_revenue,

    AVG(total_item_value) AS average_item_value

FROM ecommerce_lakehouse.gold.fact_order_items

GROUP BY DATE(order_purchase_timestamp);

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE ecommerce_lakehouse.gold.product_performance
USING DELTA
AS
SELECT

    product_id,

    product_category_name_english,

    COUNT(DISTINCT order_id) AS total_orders,

    SUM(total_item_value) AS total_revenue,

    SUM(price) AS product_revenue,

    SUM(freight_value) AS freight_revenue,

    AVG(average_review_score) AS average_review_score,

    SUM(order_item_id) AS item_count

FROM ecommerce_lakehouse.gold.fact_order_items

GROUP BY
    product_id,
    product_category_name_english;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE ecommerce_lakehouse.gold.customer_360
USING DELTA
AS
SELECT

    customer_id,

    COUNT(DISTINCT order_id) AS total_orders,

    SUM(total_item_value) AS total_spend,

    AVG(total_item_value) AS average_order_value,

    AVG(average_review_score) AS average_review_score,

    MAX(order_purchase_timestamp) AS last_order_date,

    MIN(order_purchase_timestamp) AS first_order_date,

    COUNT(DISTINCT product_id) AS unique_products,

    COUNT(DISTINCT product_category_name_english)
        AS unique_categories

FROM ecommerce_lakehouse.gold.fact_order_items

GROUP BY customer_id;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE ecommerce_lakehouse.gold.seller_performance
USING DELTA
AS
SELECT

    seller_id,

    seller_state,

    COUNT(DISTINCT order_id) AS total_orders,

    COUNT(*) AS total_items,

    SUM(price) AS product_revenue,

    SUM(freight_value) AS freight_revenue,

    SUM(total_item_value) AS total_revenue,

    AVG(average_review_score) AS average_review_score

FROM ecommerce_lakehouse.gold.fact_order_items

GROUP BY
    seller_id,
    seller_state;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE ecommerce_lakehouse.gold.payment_analysis
USING DELTA
AS
SELECT

    primary_payment_type AS payment_type,

    COUNT(DISTINCT order_id) AS total_orders,

    SUM(total_payment_value) AS total_payment_value,

    AVG(total_payment_value) AS average_payment_value,

    AVG(max_installments) AS average_installments

FROM ecommerce_lakehouse.gold.fact_order_items

WHERE primary_payment_type IS NOT NULL

GROUP BY primary_payment_type;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE ecommerce_lakehouse.gold.delivery_performance
USING DELTA
AS
SELECT

    order_id,

    order_status,

    order_purchase_timestamp,

    order_delivered_customer_date,

    order_estimated_delivery_date,

    CASE
        WHEN order_delivered_customer_date IS NULL
        THEN NULL

        ELSE DATEDIFF(
            DATE(order_delivered_customer_date),
            DATE(order_estimated_delivery_date)
        )
    END AS delivery_difference_days,

    CASE
        WHEN order_delivered_customer_date IS NULL
        THEN 'NOT_DELIVERED'

        WHEN order_delivered_customer_date
             <= order_estimated_delivery_date
        THEN 'ON_TIME'

        ELSE 'LATE'
    END AS delivery_status

FROM ecommerce_lakehouse.silver.orders;

num_affected_rows,num_inserted_rows
